First we import the libraries we will use for our recommender system.

In [36]:
import pandas as pd 
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Loading our dataset

In [37]:
hostels = pd.read_csv("hostel_data.csv")

#Displaying the first five rows of the dataframe
print("Hostels DataFrame:")
print(hostels.head(5))

Hostels DataFrame:
   S/N   HOSTEL NAME ROOM TYPE  PRICE (GHS)  \
0    1  HAPPY FAMILY    4 in 1         2600   
1    1  HAPPY FAMILY    3 in 1         5900   
2    1  HAPPY FAMILY    2 in 1         7400   
3    2    ABC HOSTEL    4 in 1         3800   
4    2    ABC HOSTEL    2 in 1         6200   

   DISTANCE FROM CAMPUS(min walk to Campus)  
0                                         9  
1                                         9  
2                                         9  
3                                        11  
4                                        11  


In [38]:
hostels.columns = ["S/N", "Hostel_Name", "Room_Type", "Price", "Distance"]

# Clean text
def clean_text(x):
    return str(x).lower().replace(" ", "")

hostels["Room_Type"] = hostels["Room_Type"].apply(clean_text)

# Price bucket based on room type
def price_bucket(row):
    price = row["Price"]
    room = row["Room_Type"]

    if "1" in room or "one" in room or "single" in room:
      if price < 7800:   return "low"
      elif price < 8600: return "medium"
      else:              return "high"
    elif "2" in room or "two" in room or "double" in room:
        if price < 6800:    return "low"
        elif price < 7800: return "medium"
        else:               return "high"
    elif "3" in room or "three" in room or "triple" in room:
        if price < 5600:   return "low"
        elif price < 7200: return "medium"
        else:              return "high"
    else:  # 4-in-a-room or unknown
        if price < 3000:   return "low"
        elif price < 5800: return "medium"
        else:              return "high"

hostels["Price_Range"] = hostels.apply(price_bucket, axis=1)

# Distance bucket in walking minutes
def distance_bucket(mins):
    if mins <= 15:    return "near"
    elif mins <= 30: return "mid"
    else:            return "far"

hostels["Distance_Range"] = hostels["Distance"].apply(distance_bucket)

# Create soup
def create_soup(x):
    return x["Room_Type"] + " " + x["Price_Range"] + " " + x["Distance_Range"]

hostels["soup"] = hostels.apply(create_soup, axis=1)

# Vectorize
count = CountVectorizer(stop_words="english")
count_matrix = count.fit_transform(hostels["soup"])

# Cosine similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)

# Reset index
hostels = hostels.reset_index(drop=True)
indices = pd.Series(hostels.index, index=hostels["Hostel_Name"])

# Recommendation function
def get_filtered_recommendations(name, cosine_sim, max_price=None, max_distance=None, top_n=10):
    raw_idx = indices[name]
    idx = raw_idx.iloc[0] if isinstance(raw_idx, pd.Series) else raw_idx

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    hostel_indices = [i[0] for i in sim_scores[1:]]
    candidates = hostels.iloc[hostel_indices]

    if max_price is not None:
        candidates = candidates[candidates["Price"] <= max_price]
    if max_distance is not None:
        candidates = candidates[candidates["Distance"] <= max_distance]

    return candidates[["Hostel_Name", "Room_Type", "Price", "Distance"]].head(top_n)

# Test
print(get_filtered_recommendations("GEORGIA HOSTEL", cosine_sim, max_price=6000, max_distance=10))

               Hostel_Name Room_Type  Price  Distance
0             HAPPY FAMILY      4in1   2600         9
5   NANA ADOMA MAIN HOSTEL      4in1   2800         3
12      WHITE HOUSE HOSTEL      4in1   5200         5
16          AMANDAH HOSTEL      4in1   5800         6
25           UNIQUE HOSTEL      4in1   4400        10
32                  PENIEL      4in1   4000        10
37            FRANCO ANNEX      4in1   3700        10
41         NO WEAPON ANNEX      4in1   4000         9
45                EBENEZER      4in1   4200         9
70                 ANAROSA      4in1   5600        10


For our Hostel Project, Unlike Netflix's similarity alorithm we are recommending hostels based on parameters type, price and distance, so with the function below will accomplish that.

In [39]:
def hostel_to_text(row):
    return f"{row['Hostel_Name']} ({row['Room_Type']}) - GHS {row['Price']}, {row['Distance']}km from campus"

In [40]:
def recommend_by_preferences(room_type=None, max_price=None, max_distance=None, top_n=5):
    candidates = hostels.copy()

    # Filter by room type if provided
    if room_type is not None:
        room_type_clean = room_type.lower().replace(" ", "")
        candidates = candidates[candidates["Room_Type"] == room_type_clean]

    # Filter by max price if provided
    if max_price is not None:
        candidates = candidates[candidates["Price"] <= max_price]

    # Filter by max distance if provided
    if max_distance is not None:
        candidates = candidates[candidates["Distance"] <= max_distance]

    # Sort by price then distance so best value shows first
    candidates = candidates.sort_values(by=["Price", "Distance"], ascending=True)

    if candidates.empty:
        print("No hostels found matching your criteria.")
        return None

    return candidates[["Hostel_Name", "Room_Type", "Price", "Distance"]].head(top_n)

#Example
print(recommend_by_preferences(room_type="3in1", max_price=6000, max_distance=30))

              Hostel_Name Room_Type  Price  Distance
113       THE BEST HOSTEL      3in1   3400        30
18   SHALOM KIBUTZ HOSTEL      3in1   3500         6
546       AMERICAN HOSTEL      3in1   3600         4
459               MAJESTY      3in1   3600        14
584         ENYIMA HOSTEL      3in1   3600        14


# Using Groq (Natural Language Understanding) to allow for less rigid prompts to the program

In [41]:
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
from groq import Groq
import json

client = Groq(api_key="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")

def extract_filters_with_ai(user_message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """You are a hostel search assistant for a university in Ghana.
                Extract search filters and return ONLY a JSON object with these keys:
                - room_type: one of '1in1','2in1','3in1','4in1', or null
                - max_price: a number in GHS or null
                - max_distance: a number in WALKING MINUTES to campus or null

                Distance guidance:
                - "close", "near", "walking distance" → max_distance: 15
                - "not too far", "fairly close"        → max_distance: 30
                - "far is okay", no mention of distance → max_distance: null

                Price guidance (set max_price based on BOTH budget word AND room type together):
                - 1in1: cheap → 7800  | mid → 8600  | any/unmentioned → null
                - 2in1: cheap → 6800  | mid → 7800 | any/unmentioned → null
                - 3in1: cheap → 5600  | mid → 7200  | any/unmentioned → null
                - 4in1: cheap → 3000  | mid → 5800  | any/unmentioned → null
                - room type unknown: cheap → 4000   | mid → 7800  | any/unmentioned → null

                Examples:
                Input: "single room under 8000 cedis close to campus"
                Output: {"room_type": "1in1", "max_price": 7999, "max_distance": 15}

                Input: "show me all hostels"
                Output: {"room_type": null, "max_price": null, "max_distance": null}

                Input: "cheap 2 in a room not too far"
                Output: {"room_type": "2in1", "max_price": 6800, "max_distance": 30}

                Input: "4 in a room, price doesn't matter, close by"
                Output: {"room_type": "4in1", "max_price": null, "max_distance": 15}

                Input: "cheap 3 in a room not too far"
                Output: {"room_type": "3in1", "max_price": 5600, "max_distance": 30}

                Input: "affordable 4 in a room close to campus"
                Output: {"room_type": "4in1", "max_price": 4000, "max_distance": 15}

                Input: "mid range single room"
                Output: {"room_type": "1in1", "max_price": 8000, "max_distance": null}

                Input: "cheap room, don't care about type"
                Output: {"room_type": null, "max_price": 4000, "max_distance": null}

                Input: "show me all hostels"
                Output: {"room_type": null, "max_price": null, "max_distance": null}


                Return ONLY the JSON, no extra text, no explanation."""
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("Could not parse AI response:", text)
        return {"room_type": None, "max_price": None, "max_distance": None}

print("Groq AI ready!")

Groq AI ready!


In [43]:
def ai_chatbot(user_message):
    print(f"\nYou: {user_message}")

    filters = extract_filters_with_ai(user_message)

    room_type    = filters.get("room_type")
    max_price    = filters.get("max_price")
    max_distance = filters.get("max_distance")


    summary = []
    if room_type:     summary.append(f"room type: {room_type}")
    if max_price:     summary.append(f"max price: GHS {max_price}")
    if max_distance:  summary.append(f"max distance: {max_distance} min walk")  # ← was "km"

    if summary:
        print(f"Bot: Understood! Searching for — {', '.join(summary)}\n")
    else:
        print("Bot: No filters detected, showing top hostels.\n")

    return recommend_by_preferences(
        room_type=room_type,
        max_price=max_price,
        max_distance=max_distance
    )
    if result is not None and len(result) > 0:
        text = "\n\n".join(result.apply(hostel_to_text, axis=1))
    else:
        text = "No hostels found matching your criteria."

    print(f"Bot:\n{text}")

    return text

In [44]:
ai_chatbot("I want a room with two people under 6000 cedis close to campus")


You: I want a room with two people under 6000 cedis close to campus
Bot: Understood! Searching for — room type: 2in1, max price: GHS 5999, max distance: 15 min walk



,Hostel_Name,Room_Type,Price,Distance
554,AMA ODE,2in1,3600,14
592,WILLCHRIS HOSTEL,2in1,3600,14
709,KODUA'S HOSTEL,2in1,3700,14
104,THY WILL BE DONE,2in1,3800,14
632,VIC HOSTEL,2in1,3800,14
